# **Step 1: : Load everything from hierarchy**

In [2]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np

drive.mount('/content/drive')
DRIVE     = Path('/content/drive/MyDrive/m5_project')
PROCESSED = DRIVE / 'processed'

%run /content/drive/MyDrive/m5_project/notebooks/hierarchy.ipynb

cols = pd.read_parquet(PROCESSED / 'ca12_long.parquet').columns.tolist()
print(cols)

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(6096944, 8)
days: 847 -> 1941
L1_total            1 series
L2_store            2 series
L3_cat              3 series
L4_dept             7 series
L5_store_cat        6 series
L6_store_dept      14 series
L7_item          3049 series
L8_item_store    6098 series
(6098, 1095)
L1_total            1 series x 1095 days
L2_store            2 series x 1095 days
L3_cat              3 series x 1095 days
L4_dept             7 series x 1095 days
L5_store_cat        6 series x 1095 days
L6_store_dept      14 series x 1095 days
L7_item          3049 series x 1095 days
L8_item_store    6098 series x 1095 days
Level 1: Sum of both the stores
d       847   848   849   850   851
Total  7745  8747  7996  8197  6266
----------------------------------------

 Level 2: Sum of each Store
           847   848   849   850   851
store_id                    

# **Step 2 Calendar table**




In [3]:
cal = (pd.read_parquet(PROCESSED / 'ca12_long.parquet',
                       columns=['d', 'date', 'wday', 'month', 'event_type_1', 'snap_CA'])
         .drop_duplicates('d').set_index('d').sort_index())

cal['day_of_month'] = pd.to_datetime(cal['date']).dt.day
cal['event_type']   = cal['event_type_1'].astype('category').cat.codes      # event kind as a number
cal = cal[['wday', 'month', 'day_of_month', 'event_type', 'snap_CA']]

print(cal.shape)            # expect (1095, 5): one row per day, d_847..d_1941
cal.loc[1914:1920]          # first week of the test period

(1095, 5)


,wday,month,day_of_month,event_type,snap_CA
d,,,,,
1914,3,4,25,-1,0
1915,4,4,26,-1,0
1916,5,4,27,-1,0
1917,6,4,28,-1,0
1918,7,4,29,-1,0
1919,1,4,30,2,0
1920,2,5,1,2,1


In [7]:
print(dict(enumerate(pd.read_parquet(PROCESSED / 'ca12_long.parquet', columns=['event_type_1'])['event_type_1'].astype('category').cat.categories)))

# event codes

{0: 'Cultural', 1: 'National', 2: 'Religious', 3: 'Sporting'}


# **Step 3: Past Sell features**

- The best clue about tomorrow’s sales is past sales. This cell gives each day two kinds of clues:

- lags: what sold exactly x days earlier.
- rolling means: the average over the last y days, counting back from 28 days ago